# 🎙️ Clone My Voice — Chatterbox (free)

Make video narration in **your own voice**, free, on Colab's GPU. Audio is saved to your Google Drive.

**Before you start:** `Runtime → Change runtime type → T4 GPU`, then run the cells top to bottom (▶ or Shift+Enter).

You need a **10–30 second recording of your own voice** (quiet room, no music, natural speaking).

> Only clone your own voice or one you have permission to use. Chatterbox adds an inaudible AI watermark to every clip.

## 1. Check the GPU

In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — switch runtime to T4 GPU (will still work on CPU, slowly)")

## 2. Connect Google Drive
Your voice sample and narrations are kept in `My Drive/VoiceClone/`.

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/VoiceClone'
os.makedirs(f'{WORK}/output', exist_ok=True)
print('Folder:', WORK)

## 3. Install Chatterbox (~3–5 min, once per session)
If it ends with a message asking to **restart the session**, do `Runtime → Restart session`, then skip to step 4 (re-run step 2 too).

In [ ]:
%cd /content
!test -d chatterbox || git clone -q -b claude/zealous-johnson-m3294m https://github.com/gayanabeysekara-bot/chatterbox.git
%cd /content/chatterbox
!pip install -q -e .
print('✅ Installed')

## 4. Your voice sample
Uses `My Drive/VoiceClone/my_voice.wav` if it's there (saved from a previous session). Otherwise you'll be asked to upload one (WAV or MP3).

In [ ]:
import os, shutil, sys
sys.path.insert(0, '/content/chatterbox/src')
WORK = '/content/drive/MyDrive/VoiceClone'
VOICE = None
for name in ('my_voice.wav', 'my_voice.mp3'):
    if os.path.exists(f'{WORK}/{name}'):
        VOICE = f'{WORK}/{name}'
if VOICE is None:
    from google.colab import files
    up = files.upload()
    src = next(iter(up))
    VOICE = f'{WORK}/my_voice' + os.path.splitext(src)[1].lower()
    shutil.move(src, VOICE)
print('Voice sample:', VOICE)
from IPython.display import Audio, display
display(Audio(VOICE))

## 5. Load the model (once per session)

In [ ]:
import torch
from chatterbox.tts_turbo import ChatterboxTurboTTS
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = ChatterboxTurboTTS.from_pretrained(device=device, nano=(device == 'cpu'))
print('✅ Model loaded on', device)

## 6. Write your script and generate
- Put a **blank line between paragraphs** — each is generated separately for stability.
- You can add tags like `[laugh]`, `[chuckle]`, `[sigh]`.
- Change `FILE_NAME` for each video. Re-run this cell as often as you like.

In [ ]:
FILE_NAME = 'video_01_narration'   # output name (no extension)
PAUSE_SECONDS = 0.4                # silence between paragraphs

SCRIPT = """
Hi everyone, welcome back to the channel!

Today I'm going to show you something really interesting. [chuckle] Stick around till the end.
"""

import torch, torchaudio as ta
from IPython.display import Audio, display
chunks = [p.strip() for p in SCRIPT.split('\n\n') if p.strip()]
silence = torch.zeros(1, int(model.sr * PAUSE_SECONDS))
parts = []
for i, chunk in enumerate(chunks, 1):
    print(f'[{i}/{len(chunks)}] {chunk[:70]}')
    parts += [model.generate(chunk, audio_prompt_path=VOICE).cpu(), silence]
wav = torch.cat(parts, dim=1)
out = f'{WORK}/output/{FILE_NAME}.wav'
ta.save(out, wav, model.sr)
print('✅ Saved to Drive:', out)
display(Audio(wav.numpy()[0], rate=model.sr))

## 7. (Optional) Also save as MP3

In [ ]:
mp3 = out[:-4] + '.mp3'
!ffmpeg -y -loglevel error -i "{out}" -b:a 192k "{mp3}"
print('✅ Saved:', mp3)

## Tips
- **Voice doesn't sound like you?** Use a cleaner, longer (20–30 s) sample with your normal speaking tone. Replace `my_voice.wav` in Drive and re-run step 4.
- **Next time:** just run steps 1–5, then step 6 — your voice sample is already saved in Drive.